# Athlete2Humanoid: Day 1 setup

Goal for today: every piece of the pipeline makes **one successful call**.

| # | Check | Done when |
|---|---|---|
| 1 | G1 loads in MuJoCo and renders | `g1_stand.mp4` plays below |
| 2 | mink IK works on G1 | `g1_ik_squat.mp4` shows a squat |
| 3 | Muse Spark text call | prints a reply |
| 4 | Muse Spark image call | answers a question about a robot frame (tells you if images work) |
| 5 | SAM 3.1 tracks "athlete" | `sam_overlay.mp4` shows your athlete tinted red |

**Before you start:** Runtime → Change runtime type → **L4 GPU** (or A100). T4 is too old for SAM 3.1's attention kernels.
Add two Colab secrets (🔑 key icon in the left sidebar, toggle *Notebook access* on):
`META_API_KEY` (Meta developer console) and `HF_TOKEN` (Hugging Face token, *after* your access to `facebook/sam3.1` is approved).

## 0. Install (run once per runtime)

In [ ]:
# Where your code lives. Option A (recommended): push the repo to GitHub and paste the URL.
# Option B: leave REPO_URL empty and upload the athlete2humanoid folder to Drive at DRIVE_CODE.
REPO_URL = ""  # e.g. "https://github.com/<you>/athlete2humanoid.git"

from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, importlib.metadata
DRIVE_ROOT = "/content/drive/MyDrive/a2h"
DRIVE_CODE = f"{DRIVE_ROOT}/athlete2humanoid"
REPO = "/content/athlete2humanoid" if REPO_URL else DRIVE_CODE
os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/clips", exist_ok=True)

if REPO_URL and not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
assert os.path.exists(f"{REPO}/config.py"), f"Code not found at {REPO}"

numpy_before = importlib.metadata.version("numpy")
!pip install -q -r {REPO}/requirements.txt
# SAM 3.1 from source (the 3.1 multiplex predictor needs the latest code).
if not os.path.exists("/content/sam3"):
    !git clone -q https://github.com/facebookresearch/sam3.git /content/sam3
!pip install -q -e "/content/sam3[notebooks]"
!bash {REPO}/scripts/fetch_assets.sh

if importlib.metadata.version("numpy") != numpy_before:
    print(f"numpy changed {numpy_before} -> {importlib.metadata.version('numpy')}: restarting the runtime.")
    print("When it reconnects, skip this cell and run from step 1.")
    os.kill(os.getpid(), 9)

## 1. Session setup (run after every restart)

In [ ]:
REPO_URL = ""  # same value as above
import os, sys
os.environ["MUJOCO_GL"] = "egl"   # headless rendering; must be set before importing mujoco
DRIVE_ROOT = "/content/drive/MyDrive/a2h"
REPO = "/content/athlete2humanoid" if REPO_URL else f"{DRIVE_ROOT}/athlete2humanoid"
os.environ["A2H_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"   # outputs survive disconnects
os.environ["A2H_CLIPS_DIR"] = f"{DRIVE_ROOT}/clips"
os.environ["A2H_ASSETS_DIR"] = f"{REPO}/assets"
os.chdir(REPO); sys.path.insert(0, REPO)

from google.colab import userdata
for k in ("META_API_KEY", "HF_TOKEN"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception as e:
        print(f"⚠️  secret {k} missing ({type(e).__name__}); steps that need it will fail")

import mediapy, numpy as np, torch
import config
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("outputs ->", config.OUTPUT_DIR)

## 2. G1 in MuJoCo

In [ ]:
import render
model, data = render.load_g1()
print(f"G1: {model.nq} qpos, {model.nu} actuators, dt={model.opt.timestep}")
frames = render.render_physics(model, data, seconds=3.0)
print(f"pelvis height after 3 s of physics: {data.qpos[2]:.2f} m (≈0.79 means still standing)")
render.save_video(frames, config.OUTPUT_DIR / "g1_stand.mp4")
mediapy.show_video(frames, fps=30)

## 3. mink IK smoke test (derisks day 3)
Feet pinned, pelvis lowered 25 cm and back up. Should print errors under ~1 cm.

In [ ]:
!python retarget.py
mediapy.show_video(mediapy.read_video(str(config.OUTPUT_DIR / "g1_ik_squat.mp4")), fps=30)

## 4. Muse Spark: text
If this 404s or says the model doesn't exist, check the base URL and model name in the Meta developer console
and set `META_API_BASE` / `MUSE_MODEL` (see `config.py`).

In [ ]:
import coach
print(coach.ask("In one sentence: what are the phases of a vertical squat jump?"))

## 5. Muse Spark: image input (answers a kickoff question)

In [ ]:
frame_path = config.OUTPUT_DIR / "g1_frame.png"
mediapy.write_image(str(frame_path), frames[-1])
try:
    print(coach.ask("Describe this robot's posture in one sentence. Is it standing upright?", images=[frame_path]))
    print("\n✅ Images work → on day 4 you can send video frames.")
except Exception as e:
    print(f"❌ Image input failed ({e}).\n→ Plan B for day 4: send numeric joint-angle summaries.")

## 6. SAM 3.1: track the athlete
Put your 2–4 s clip in Drive at `a2h/clips/raw.mp4` (or set `RAW` below). Until you have one, the cell falls back to SAM's sample video.
The first run downloads ~3.5 GB of weights.

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

RAW = config.CLIPS_DIR / "raw.mp4"
START, DURATION = 0.0, 4.0                 # seconds to keep
CLIP = config.CLIPS_DIR / "clip.mp4"
if RAW.exists():
    # Trim, 30 fps, 720p height: keeps SAM fast and gives MediaPipe enough pixels.
    !ffmpeg -y -loglevel error -ss {START} -t {DURATION} -i "{RAW}" -vf "fps=30,scale=-2:720" -an "{CLIP}"
    prompt = "athlete"
else:
    print("No raw.mp4 yet: using SAM's sample video with prompt 'person'.")
    CLIP, prompt = "/content/sam3/assets/videos/bedroom.mp4", "person"

import track
video = mediapy.read_video(str(CLIP))
outputs = track.track(CLIP, prompt=prompt)
masks, obj_id = track.main_track_masks(outputs, len(video))
print(f"{len(video)} frames, athlete = object {obj_id}, visible in {masks.any((1,2)).mean():.0%} of frames")
vis = track.overlay(np.asarray(video), masks)
render.save_video(vis, config.OUTPUT_DIR / "sam_overlay.mp4", fps=video.metadata.fps)
mediapy.show_video(vis, fps=video.metadata.fps)

## Done for day 1?
- [ ] All five checks above passed (videos are in Drive at `a2h/outputs/`)
- [ ] Wrote down kickoff answers in `DAY1_CHECKLIST.md`: Muse Spark inputs, SAM 3.1 via API or weights, submission rules
- [ ] Clip chosen and trimmed (`a2h/clips/clip.mp4`) and SAM tracks the right person through the whole clip
- [ ] Code pushed to GitHub

**Tomorrow (day 2):** `track.crop_to_mask` → `pose.landmarks_from_crops` → `pose.smooth` → save `(frames, 33, 3)`.